In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
import lightgbm as lgb
import features as F

train = pd.read_csv("../data/train_test.csv")
val = pd.read_csv("../data/validation.csv")
train["rpm"] = train["posted_rate"] / train["distance"]

print("Training data:  ", train.shape, "|", train["date"].min(), "to", train["date"].max())
print("Validation data:", val.shape, "|", val["date"].min(), "to", val["date"].max())

clean = F.drop_bad_labels(train)
print("Correlation of distance with rate, raw:    ", round(train["distance"].corr(train["posted_rate"]), 3))
print("Correlation of distance with rate, cleaned:", round(clean["distance"].corr(clean["posted_rate"]), 3))
print()
print("Median rate per mile by equipment:")
print(clean.groupby("equipment")["rpm"].median().round(2))

Training data:   (48000, 15) | 2025-01-01 to 2025-10-31
Validation data: (12000, 13) | 2025-11-01 to 2025-12-31
Correlation of distance with rate, raw:     0.909
Correlation of distance with rate, cleaned: 0.982

Median rate per mile by equipment:
equipment
Dry Van    2.04
Flatbed    2.22
Reefer     2.31
Name: rpm, dtype: float64


In [2]:
bucket = pd.cut(train["distance"], F.BUCKETS)
ratio = train["rpm"] / train.groupby(bucket, observed=True)["rpm"].transform("median")

print("Corrupted rates (more than 2x the typical rate per mile):", (ratio > 2).sum())
print("Negative weights:", (train["weight"] < 0).sum())
print("Missing weight:", train["weight"].isna().sum())
print()
print("Rows by ratio to the typical rate per mile:")
print(pd.cut(ratio, [0, 1.5, 2, 2.5, 3, 4, 6, 20]).value_counts().sort_index())

Corrupted rates (more than 2x the typical rate per mile): 340
Negative weights: 292
Missing weight: 300

Rows by ratio to the typical rate per mile:
rpm
(0.0, 1.5]     47660
(1.5, 2.0]         0
(2.0, 2.5]        36
(2.5, 3.0]        72
(3.0, 4.0]       122
(4.0, 6.0]       110
(6.0, 20.0]        0
Name: count, dtype: int64


In [3]:
dev = F.drop_bad_labels(train)
dev["month"] = pd.to_datetime(dev["date"]).dt.month
dev["week"] = pd.to_datetime(dev["date"]).dt.isocalendar().week.astype(int)

fit_raw, hold_raw = dev[dev["month"] <= 8], dev[dev["month"] >= 9]
print("Train on Jan-Aug:", len(fit_raw), "rows")
print("Test on Sep-Oct: ", len(hold_raw), "rows")

Train on Jan-Aug: 38210 rows
Test on Sep-Oct:  9450 rows


In [4]:
early, aug = dev[dev["week"] < 32], dev[(dev["week"] >= 32) & (dev["week"] <= 35)]

def score(fit_raw, hold_raw, cols):
    med = F.weight_medians(fit_raw)
    fit, hold = F.add_features(fit_raw, med), F.add_features(hold_raw, med)
    m = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.03, num_leaves=31,
                          min_child_samples=40, subsample=0.8, subsample_freq=1,
                          random_state=42, verbose=-1)
    m.fit(fit[cols], fit["rpm"])
    pred = m.predict(hold[cols]) * hold["distance"]
    return round((pred - hold["posted_rate"]).abs().mean(), 1)

basic = ["distance", "eq", "weight"]
sets = {
    "Distance, equipment, weight": basic,
    "Final features": F.FEATURES,
    "Final features + quote_signal": F.FEATURES + ["quote_signal"],
    "Distance, equipment, weight + quote_signal": basic + ["quote_signal"],
}
print(f"{'MAE in dollars':45s}{'Test 1 (Sep-Oct)':>18s}{'Test 2 (August)':>18s}")
for name, cols in sets.items():
    print(f"{name:45s}{score(fit_raw, hold_raw, cols):>18}{score(early, aug, cols):>18}")

MAE in dollars                                 Test 1 (Sep-Oct)   Test 2 (August)
Distance, equipment, weight                                73.1              67.9
Final features                                             58.4              50.2
Final features + quote_signal                              48.8             109.3
Distance, equipment, weight + quote_signal                 52.1             121.6
